# Quick claude converter for turning compartment_mask.tif files into geojsons, in case you didn't initially use geojsons...

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Load, convert & plot                                             ║
# ║  Run this first to see what grayscale values exist in your mask.           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
 
import json
import numpy as np
import tifffile as tf
import matplotlib.pyplot as plt
from shapely.geometry import shape, mapping
from rasterio.features import shapes
from rasterio.transform import from_bounds
from pathlib import Path
 
# ── SET THESE ─────────────────────────────────────────────────────────────────
INPUT_PATH  = "path/to/treated_mask.tif"   # <-- your .tif file
OUTPUT_DIR  = "path/to/compartment_segmentations" # folder for saved GeoJSONs
# ──────────────────────────────────────────────────────────────────────────────
 
def mask_to_geojson(mask, value):
    """Vectorise one grayscale value → GeoJSON FeatureCollection (disconnected components supported)."""
    binary    = (mask == value).astype(np.uint8)
    h, w      = binary.shape
    transform = from_bounds(0, h, w, 0, w, h)
    features  = []
    for geom, val in shapes(binary, mask=binary, transform=transform):
        if val == 1:
            poly = shape(geom)
            if poly.is_valid and not poly.is_empty:
                features.append({
                    "type": "Feature",
                    "properties": {"grayscale_value": int(value)},
                    "geometry": mapping(poly)
                })
    return {"type": "FeatureCollection", "features": features}
 
# Load mask
mask          = tf.imread(INPUT_PATH)
unique_values = sorted(int(v) for v in np.unique(mask) if v != 0)
print(f"Unique non-background values: {unique_values}")
 
# Convert every value and plot it for review
geojson_dict = {}
for value in unique_values:
    geojson_dict[value] = mask_to_geojson(mask, value)
 
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(mask == value, cmap="Blues", interpolation="nearest")
    n = len(geojson_dict[value]["features"])
    ax.set_title(f"Grayscale value: {value}  |  {n} component(s)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Rename keys & save                                               ║
# ║  Edit LABEL_MAP below based on the plots above, then run this cell.        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
 
# Map each grayscale value to a human-readable name.
# Use the values printed by Cell 1. Any value not listed here keeps its number.
LABEL_MAP = {
    1:  "Epidermis",
    2: "Cartilage",
    3: "Dermis",
    4: 'Hair_Follicles'
}
 
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
 
for value, geojson in geojson_dict.items():
    label     = LABEL_MAP.get(value, str(value))
    safe_name = label.replace(" ", "_").replace("/", "-")
    out_path  = output_dir / f"{safe_name}.geojson"
    with open(out_path, "w") as f:
        json.dump(geojson, f, indent=2)
    print(f"  {value} → '{label}'  ({len(geojson['features'])} component(s))  →  {out_path}")
 
print("\nDone.")